# WeedyRice: three modalities, seeds 123 and 2026

Self-contained runner embedding the original training scripts. Saved output stops during the final proposed-model run, so it is not a complete final-run archive.

Source: `gd-second.ipynb`, zero-based cell 11.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""
Standalone Kaggle runner for WeedyRice-RGBMS-DB.

Runs ONLY:
  Seed 123  -> RGB-only, MS-only, Proposed RGB+MS
  Seed 2026 -> RGB-only, MS-only, Proposed RGB+MS

No old model, checkpoint, or previous script is required.
Everything is embedded in this one file.

Kaggle-safe changes:
- DataLoader num_workers = 0
- tqdm progress bars disabled
- RGB/MS baselines: max 50 epochs, patience 12
- Proposed fusion: full setting, max 70 epochs, patience 12
"""

from pathlib import Path
import subprocess
import sys
import re
import pandas as pd

SEEDS = [123, 2026]
WORK = Path("/kaggle/working")

BASELINE_SOURCE = '\n"""\nWeedyRice-RGBMS-DB\nRGB-only and MS-only baseline training on the OFFICIAL split.\n\nPurpose:\n- Replicate the same unimodal comparison used in the GD study.\n- Compare:\n    1) RGB-only U-Net: R, G, B\n    2) MS-only U-Net : Green, Red-Edge, NIR\n- Same official train/val/test split as the proposed WeedyRice run.\n- Same 896x672 input size, seed 42, optimizer, loss, threshold-selection protocol.\n- Fresh training from scratch for each modality.\n\nThe dataset\'s MS Red band is intentionally not used in this controlled transfer experiment,\nso MS-only uses the same three spectral channels as the GD manuscript:\nGreen + Red-Edge + NIR.\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport random\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom tqdm.auto import tqdm\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\n\n\n# =============================================================================\n# 1. CONFIG\n# =============================================================================\n\nROOT = Path("/kaggle/input/datasets/hemal7/weedy-rgb-ms")\n\n# Find actual dataset folder automatically\ntrain_list_candidates = list(ROOT.rglob("train_list.txt"))\nif not train_list_candidates:\n    raise FileNotFoundError(f"Could not find train_list.txt under {ROOT}")\n\nDATA_DIR = train_list_candidates[0].parent\n\nRGB_DIR = DATA_DIR / "RGB"\nMS_DIR = DATA_DIR / "Multispectral"\nMASK_DIR = DATA_DIR / "Masks"\n\nfor p in (RGB_DIR, MS_DIR, MASK_DIR):\n    if not p.exists():\n        raise FileNotFoundError(f"Missing required folder: {p}")\n\nWORK_DIR = Path("/kaggle/working")\nOUT_DIR = WORK_DIR / "weedy_rgb_ms_unimodal_seed42"\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nSEED = 42\n\n# Run both baselines\nMODALITIES = ["RGB", "MS"]\n\n# Use full settings for fair comparison with the proposed run\nMAX_EPOCHS = 70\nEARLY_STOPPING_PATIENCE = 12\n\nTARGET_SIZE = (896, 672)  # width, height\nBATCH_SIZE = 2\nACCUMULATION_STEPS = 2\nLEARNING_RATE = 1e-4\nWEIGHT_DECAY = 1e-4\nNUM_WORKERS = 2\n\n# Plain U-Net used for unimodal baselines\nUNET_CHANNELS = [32, 64, 128, 256]\n\nCOARSE_THRESHOLDS = np.round(np.arange(0.20, 0.86, 0.05), 2).tolist()\nFINE_THRESHOLDS = np.round(np.arange(0.20, 0.801, 0.01), 2).tolist()\n\nCACHE_DATA = True\n\ntry:\n    BILINEAR = Image.Resampling.BILINEAR\n    NEAREST = Image.Resampling.NEAREST\nexcept AttributeError:\n    BILINEAR = Image.BILINEAR\n    NEAREST = Image.NEAREST\n\n\ndef seed_everything(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\nseed_everything(SEED)\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\npin_memory = device.type == "cuda"\n\nprint("=" * 100)\nprint("WEEDY RICE — RGB-ONLY AND MS-ONLY BASELINES")\nprint("=" * 100)\nprint("ROOT       :", ROOT)\nprint("DATA_DIR   :", DATA_DIR)\nprint("Device     :", device)\nprint("Modalities :", MODALITIES)\nprint("Seed       :", SEED)\nprint("Max epochs :", MAX_EPOCHS)\nprint("Patience   :", EARLY_STOPPING_PATIENCE)\nprint("Input size :", TARGET_SIZE)\n\n\n# =============================================================================\n# 2. OFFICIAL SPLITS\n# =============================================================================\n\ndef read_list(path: Path):\n    with open(path, "r", encoding="utf-8") as f:\n        return [line.strip() for line in f if line.strip()]\n\n\ntrain_names = read_list(DATA_DIR / "train_list.txt")\nval_names = read_list(DATA_DIR / "val_list.txt")\ntest_names = read_list(DATA_DIR / "test_list.txt")\n\nprint("\\nOfficial splits:")\nprint("Train:", len(train_names))\nprint("Val  :", len(val_names))\nprint("Test :", len(test_names))\nprint("Total:", len(train_names) + len(val_names) + len(test_names))\n\nassert len(set(train_names) & set(val_names)) == 0\nassert len(set(train_names) & set(test_names)) == 0\nassert len(set(val_names) & set(test_names)) == 0\n\n\n# =============================================================================\n# 3. DATA LOADING\n# =============================================================================\n\ndef filename_stem(name: str) -> str:\n    return Path(name).stem\n\n\ndef image_to_unit_float(path: Path, resize_size=None, resample=BILINEAR):\n    with Image.open(path) as img:\n        if resize_size is not None:\n            img = img.resize(resize_size, resample)\n        arr = np.asarray(img).copy()\n\n    original_dtype = arr.dtype\n\n    if np.issubdtype(original_dtype, np.integer):\n        arr = arr.astype(np.float32) / float(np.iinfo(original_dtype).max)\n    else:\n        arr = arr.astype(np.float32)\n        vmax = float(np.nanmax(arr)) if arr.size else 1.0\n        if vmax > 1.0:\n            arr = arr / max(vmax, 1e-6)\n\n    return arr\n\n\ndef load_rgb(path: Path):\n    with Image.open(path) as img:\n        img = img.convert("RGB").resize(TARGET_SIZE, BILINEAR)\n        rgb = np.asarray(img, dtype=np.float32).copy() / 255.0\n    return rgb\n\n\ndef load_single_band(path: Path):\n    arr = image_to_unit_float(path, TARGET_SIZE, BILINEAR)\n\n    if arr.ndim == 3:\n        arr = arr[..., 0]\n\n    if arr.ndim != 2:\n        raise RuntimeError(f"Expected single band, got {arr.shape}: {path}")\n\n    return arr.astype(np.float32)\n\n\ndef load_mask(path: Path):\n    with Image.open(path) as img:\n        img = img.convert("L").resize(TARGET_SIZE, NEAREST)\n        arr = np.asarray(img, dtype=np.uint8)\n\n    # Weedy rice = 255, background = 0\n    return (arr > 127).astype(np.float32)\n\n\ndef paths_for(name: str):\n    stem = filename_stem(name)\n\n    rgb_path = RGB_DIR / f"{stem}.JPG"\n    if not rgb_path.exists():\n        rgb_path = RGB_DIR / f"{stem}.jpg"\n\n    g_path = MS_DIR / f"{stem}_G.TIF"\n    re_path = MS_DIR / f"{stem}_RE.TIF"\n    nir_path = MS_DIR / f"{stem}_NIR.TIF"\n    mask_path = MASK_DIR / f"{stem}.png"\n\n    for p in (rgb_path, g_path, re_path, nir_path, mask_path):\n        if not p.exists():\n            raise FileNotFoundError(f"Missing paired file: {p}")\n\n    return rgb_path, g_path, re_path, nir_path, mask_path\n\n\ndef load_raw_6ch(name: str):\n    rgb_path, g_path, re_path, nir_path, mask_path = paths_for(name)\n\n    rgb = load_rgb(rgb_path)\n    green = load_single_band(g_path)\n    red_edge = load_single_band(re_path)\n    nir = load_single_band(nir_path)\n    mask = load_mask(mask_path)\n\n    ms = np.stack([green, red_edge, nir], axis=-1)\n    image = np.concatenate([rgb, ms], axis=-1).astype(np.float32)\n\n    if image.shape != (TARGET_SIZE[1], TARGET_SIZE[0], 6):\n        raise RuntimeError(f"Unexpected 6-channel shape {image.shape}: {name}")\n\n    return image, mask, filename_stem(name)\n\n\nprint("\\nChecking one paired sample...")\n_sample_image, _sample_mask, _sample_stem = load_raw_6ch(train_names[0])\nprint("Stem        :", _sample_stem)\nprint("6ch shape   :", _sample_image.shape)\nprint("Mask shape  :", _sample_mask.shape)\nprint("Mask values :", np.unique(_sample_mask))\n\n\n# =============================================================================\n# 4. CACHE ALL SIX CHANNELS ONCE\n# =============================================================================\n\nRAW_CACHE = {}\nall_names = train_names + val_names + test_names\n\nif CACHE_DATA:\n    print("\\nCaching resized RGB + G + RE + NIR samples...")\n    for name in tqdm(all_names, desc="Cache"):\n        image, mask, stem = load_raw_6ch(name)\n        RAW_CACHE[name] = (\n            image.astype(np.float16),\n            mask.astype(np.uint8),\n            stem,\n        )\n\n    total_gb = sum(\n        image.nbytes + mask.nbytes\n        for image, mask, _ in RAW_CACHE.values()\n    ) / (1024 ** 3)\n\n    print(f"Cached {len(RAW_CACHE)} samples (~{total_gb:.2f} GB)")\n\n\ndef get_raw_6ch(name):\n    if name in RAW_CACHE:\n        image, mask, stem = RAW_CACHE[name]\n        return image.astype(np.float32), mask.astype(np.float32), stem\n\n    return load_raw_6ch(name)\n\n\n# =============================================================================\n# 5. TRAIN-ONLY NORMALIZATION\n# =============================================================================\n\ndef compute_train_stats(names):\n    channel_sum = np.zeros(6, dtype=np.float64)\n    channel_sq = np.zeros(6, dtype=np.float64)\n    count = 0\n\n    for name in tqdm(names, desc="Train normalization"):\n        image, _, _ = get_raw_6ch(name)\n\n        channel_sum += image.sum(axis=(0, 1), dtype=np.float64)\n        channel_sq += np.square(image, dtype=np.float32).sum(\n            axis=(0, 1), dtype=np.float64\n        )\n\n        count += image.shape[0] * image.shape[1]\n\n    mean64 = channel_sum / count\n    var64 = channel_sq / count - np.square(mean64)\n\n    mean = mean64.astype(np.float32)\n    std = np.sqrt(np.maximum(var64, 1e-12)).astype(np.float32)\n\n    return mean, std\n\n\nALL_MEAN, ALL_STD = compute_train_stats(train_names)\nALL_CHANNEL_NAMES = ["R", "G", "B", "Green", "Red-Edge", "NIR"]\n\nprint("\\nTraining-set channel statistics:")\nfor ch, mean, std in zip(ALL_CHANNEL_NAMES, ALL_MEAN, ALL_STD):\n    print(f"{ch:10s} mean={mean:.6f} std={std:.6f}")\n\nwith open(OUT_DIR / "normalization_6ch.json", "w", encoding="utf-8") as f:\n    json.dump({\n        "channel_order": ALL_CHANNEL_NAMES,\n        "mean": ALL_MEAN.tolist(),\n        "std": ALL_STD.tolist(),\n        "source": "official training split only",\n    }, f, indent=2)\n\n\ndef modality_info(modality):\n    if modality == "RGB":\n        return {\n            "indices": [0, 1, 2],\n            "channels": ["R", "G", "B"],\n            "mean": ALL_MEAN[:3],\n            "std": ALL_STD[:3],\n            "input_channels": 3,\n        }\n\n    if modality == "MS":\n        return {\n            "indices": [3, 4, 5],\n            "channels": ["Green", "Red-Edge", "NIR"],\n            "mean": ALL_MEAN[3:],\n            "std": ALL_STD[3:],\n            "input_channels": 3,\n        }\n\n    raise ValueError(modality)\n\n\n# =============================================================================\n# 6. DATASET\n# =============================================================================\n\nclass WeedyUnimodalDataset(Dataset):\n    def __init__(self, names, modality, augment=False):\n        self.names = list(names)\n        self.modality = modality\n        self.augment = augment\n\n        info = modality_info(modality)\n\n        self.indices = info["indices"]\n        self.mean = np.asarray(info["mean"], dtype=np.float32).reshape(1, 1, 3)\n        self.std = np.asarray(info["std"], dtype=np.float32).reshape(1, 1, 3)\n\n    def __len__(self):\n        return len(self.names)\n\n    def __getitem__(self, index):\n        name = self.names[index]\n        image6, mask, stem = get_raw_6ch(name)\n\n        image = image6[..., self.indices].copy()\n\n        if self.augment:\n            if random.random() < 0.5:\n                image = np.flip(image, axis=1).copy()\n                mask = np.flip(mask, axis=1).copy()\n\n            if random.random() < 0.5:\n                image = np.flip(image, axis=0).copy()\n                mask = np.flip(mask, axis=0).copy()\n\n            # RGB-only: same visible perturbation used in the proposed experiment.\n            # MS-only: no radiometric perturbation.\n            if self.modality == "RGB" and random.random() < 0.75:\n                gain = np.random.uniform(\n                    0.88, 1.12, size=(1, 1, 3)\n                ).astype(np.float32)\n\n                bias = np.random.uniform(\n                    -0.035, 0.035, size=(1, 1, 3)\n                ).astype(np.float32)\n\n                image = np.clip(\n                    image * gain + bias,\n                    0.0,\n                    1.0,\n                )\n\n        image = ((image - self.mean) / self.std).astype(np.float32)\n\n        return (\n            torch.from_numpy(image.transpose(2, 0, 1)).float(),\n            torch.from_numpy(mask[None]).float(),\n            stem,\n        )\n\n\ndef make_loaders(modality, seed):\n    generator = torch.Generator()\n    generator.manual_seed(seed)\n\n    train_loader = DataLoader(\n        WeedyUnimodalDataset(train_names, modality, augment=True),\n        batch_size=BATCH_SIZE,\n        shuffle=True,\n        generator=generator,\n        num_workers=NUM_WORKERS,\n        pin_memory=pin_memory,\n        drop_last=False,\n    )\n\n    val_loader = DataLoader(\n        WeedyUnimodalDataset(val_names, modality, augment=False),\n        batch_size=1,\n        shuffle=False,\n        num_workers=NUM_WORKERS,\n        pin_memory=pin_memory,\n        drop_last=False,\n    )\n\n    test_loader = DataLoader(\n        WeedyUnimodalDataset(test_names, modality, augment=False),\n        batch_size=1,\n        shuffle=False,\n        num_workers=NUM_WORKERS,\n        pin_memory=pin_memory,\n        drop_last=False,\n    )\n\n    return train_loader, val_loader, test_loader\n\n\n# =============================================================================\n# 7. PLAIN U-NET — SAME UNIMODAL BASELINE FAMILY AS GD STUDY\n# =============================================================================\n\nclass DoubleConv(nn.Module):\n    def __init__(self, in_channels, out_channels):\n        super().__init__()\n\n        self.block = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, 3, padding=1),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(out_channels, out_channels, 3, padding=1),\n            nn.ReLU(inplace=True),\n        )\n\n    def forward(self, x):\n        return self.block(x)\n\n\nclass PlainUNet(nn.Module):\n    def __init__(self, input_channels=3, channels=UNET_CHANNELS):\n        super().__init__()\n\n        c1, c2, c3, c4 = channels\n\n        self.pool = nn.MaxPool2d(2)\n\n        self.enc1 = DoubleConv(input_channels, c1)\n        self.enc2 = DoubleConv(c1, c2)\n        self.enc3 = DoubleConv(c2, c3)\n        self.enc4 = DoubleConv(c3, c4)\n\n        self.bottleneck = DoubleConv(c4, c4 * 2)\n\n        self.up4 = nn.ConvTranspose2d(c4 * 2, c4, 2, stride=2)\n        self.dec4 = DoubleConv(c4 * 2, c4)\n\n        self.up3 = nn.ConvTranspose2d(c4, c3, 2, stride=2)\n        self.dec3 = DoubleConv(c3 * 2, c3)\n\n        self.up2 = nn.ConvTranspose2d(c3, c2, 2, stride=2)\n        self.dec2 = DoubleConv(c2 * 2, c2)\n\n        self.up1 = nn.ConvTranspose2d(c2, c1, 2, stride=2)\n        self.dec1 = DoubleConv(c1 * 2, c1)\n\n        self.out = nn.Conv2d(c1, 1, 1)\n\n    def forward(self, x):\n        e1 = self.enc1(x)\n        e2 = self.enc2(self.pool(e1))\n        e3 = self.enc3(self.pool(e2))\n        e4 = self.enc4(self.pool(e3))\n\n        b = self.bottleneck(self.pool(e4))\n\n        d4 = self.up4(b)\n        d4 = self.dec4(torch.cat([d4, e4], dim=1))\n\n        d3 = self.up3(d4)\n        d3 = self.dec3(torch.cat([d3, e3], dim=1))\n\n        d2 = self.up2(d3)\n        d2 = self.dec2(torch.cat([d2, e2], dim=1))\n\n        d1 = self.up1(d2)\n        d1 = self.dec1(torch.cat([d1, e1], dim=1))\n\n        return self.out(d1)\n\n\n# =============================================================================\n# 8. LOSS / METRICS\n# =============================================================================\n\nclass ComboLoss(nn.Module):\n    def forward(self, logits, targets):\n        targets = targets.float()\n\n        bce_map = F.binary_cross_entropy_with_logits(\n            logits,\n            targets,\n            reduction="none",\n        )\n        bce = bce_map.mean()\n\n        probability = torch.sigmoid(logits)\n\n        pt = (\n            probability * targets\n            + (1.0 - probability) * (1.0 - targets)\n        )\n\n        focal = (\n            bce_map * ((1.0 - pt) ** 2.0)\n        ).mean()\n\n        dims = tuple(range(1, logits.ndim))\n\n        intersection = (\n            probability * targets\n        ).sum(dims)\n\n        denominator = (\n            probability.sum(dims)\n            + targets.sum(dims)\n        )\n\n        dice_loss = 1.0 - (\n            (2.0 * intersection + 1e-6)\n            / (denominator + 1e-6)\n        ).mean()\n\n        return (\n            0.35 * bce\n            + 0.25 * focal\n            + 0.40 * dice_loss\n        )\n\n\ncriterion = ComboLoss()\n\n\ndef confusion_numpy(probability, mask, threshold):\n    pred = probability >= threshold\n    target = mask >= 0.5\n\n    return [\n        int(np.logical_and(pred, target).sum()),\n        int(np.logical_and(pred, ~target).sum()),\n        int(np.logical_and(~pred, target).sum()),\n        int(np.logical_and(~pred, ~target).sum()),\n    ]\n\n\ndef confusion_tensor(probability, mask, threshold):\n    pred = probability >= threshold\n    target = mask >= 0.5\n\n    return [\n        int((pred & target).sum().item()),\n        int((pred & ~target).sum().item()),\n        int((~pred & target).sum().item()),\n        int((~pred & ~target).sum().item()),\n    ]\n\n\ndef add_confusion(a, b):\n    return [x + y for x, y in zip(a, b)]\n\n\ndef calculate_metrics(values):\n    tp, fp, fn, tn = values\n    eps = 1e-7\n\n    return {\n        "dice": (2 * tp + eps) / (2 * tp + fp + fn + eps),\n        "iou": (tp + eps) / (tp + fp + fn + eps),\n        "precision": (tp + eps) / (tp + fp + eps),\n        "recall": (tp + eps) / (tp + fn + eps),\n        "accuracy": (tp + tn + eps) / (tp + fp + fn + tn + eps),\n    }\n\n\ndef save_csv(path, rows):\n    if not rows:\n        return\n\n    with open(path, "w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(\n            f,\n            fieldnames=list(rows[0].keys()),\n        )\n        writer.writeheader()\n        writer.writerows(rows)\n\n\n# =============================================================================\n# 9. TRAIN / VALIDATE HELPERS\n# =============================================================================\n\ndef train_epoch(loader, model, optimizer, scaler, epoch, modality):\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n\n    total_loss = 0.0\n\n    for step, (images, masks, _) in enumerate(\n        tqdm(loader, desc=f"{modality} train {epoch}", leave=False),\n        start=1,\n    ):\n        images = images.to(device, non_blocking=True)\n        masks = masks.to(device, non_blocking=True)\n\n        with torch.amp.autocast(\n            "cuda",\n            enabled=device.type == "cuda",\n        ):\n            logits = model(images)\n            loss = criterion(logits, masks)\n            scaled_loss = loss / ACCUMULATION_STEPS\n\n        scaler.scale(scaled_loss).backward()\n\n        if (\n            step % ACCUMULATION_STEPS == 0\n            or step == len(loader)\n        ):\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n\n            scaler.step(optimizer)\n            scaler.update()\n            optimizer.zero_grad(set_to_none=True)\n\n        total_loss += loss.item()\n\n    return total_loss / len(loader)\n\n\n@torch.no_grad()\ndef validate_model(loader, model):\n    model.eval()\n\n    totals = {\n        threshold: [0, 0, 0, 0]\n        for threshold in COARSE_THRESHOLDS\n    }\n\n    loss_sum = 0.0\n\n    for images, masks, _ in tqdm(\n        loader,\n        desc="Validation",\n        leave=False,\n    ):\n        images = images.to(device, non_blocking=True)\n        masks = masks.to(device, non_blocking=True)\n\n        with torch.amp.autocast(\n            "cuda",\n            enabled=device.type == "cuda",\n        ):\n            logits = model(images)\n            loss_sum += criterion(logits, masks).item()\n\n        probability = torch.sigmoid(logits)\n\n        for threshold in COARSE_THRESHOLDS:\n            values = confusion_tensor(\n                probability,\n                masks,\n                threshold,\n            )\n\n            totals[threshold] = add_confusion(\n                totals[threshold],\n                values,\n            )\n\n    rows = []\n\n    for threshold, values in totals.items():\n        row = calculate_metrics(values)\n        row["threshold"] = float(threshold)\n        row["loss"] = loss_sum / len(loader)\n        rows.append(row)\n\n    return max(rows, key=lambda x: x["dice"])\n\n\n@torch.no_grad()\ndef collect_predictions(loader, model, description):\n    model.eval()\n\n    records = []\n\n    for images, masks, stems in tqdm(\n        loader,\n        desc=description,\n        leave=False,\n    ):\n        images = images.to(device, non_blocking=True)\n\n        with torch.amp.autocast(\n            "cuda",\n            enabled=device.type == "cuda",\n        ):\n            probability = torch.sigmoid(model(images))\n\n        records.append({\n            "stem": stems[0],\n            "mask": masks[0, 0].numpy().astype(np.float32),\n            "probability": probability[0, 0].cpu().numpy().astype(np.float32),\n        })\n\n    return records\n\n\ndef threshold_search(records, thresholds):\n    rows = []\n\n    for threshold in thresholds:\n        total = [0, 0, 0, 0]\n\n        for record in records:\n            total = add_confusion(\n                total,\n                confusion_numpy(\n                    record["probability"],\n                    record["mask"],\n                    threshold,\n                ),\n            )\n\n        row = calculate_metrics(total)\n        row["threshold"] = float(threshold)\n        rows.append(row)\n\n    return max(rows, key=lambda x: x["dice"]), rows\n\n\n# =============================================================================\n# 10. RUN ONE MODALITY\n# =============================================================================\n\ndef run_modality(modality):\n    print("\\n" + "=" * 100)\n    print(f"RUNNING {modality}-ONLY")\n    print("=" * 100)\n\n    seed_everything(SEED)\n\n    run_dir = OUT_DIR / modality.lower()\n    run_dir.mkdir(parents=True, exist_ok=True)\n\n    train_loader, val_loader, test_loader = make_loaders(\n        modality,\n        SEED,\n    )\n\n    model = PlainUNet(input_channels=3).to(device)\n\n    parameter_count = sum(\n        p.numel()\n        for p in model.parameters()\n    )\n\n    print("Channels   :", modality_info(modality)["channels"])\n    print("Parameters :", f"{parameter_count:,}")\n\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        lr=LEARNING_RATE,\n        weight_decay=WEIGHT_DECAY,\n    )\n\n    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(\n        optimizer,\n        mode="max",\n        factor=0.5,\n        patience=4,\n        min_lr=1e-6,\n    )\n\n    scaler = torch.amp.GradScaler(\n        "cuda",\n        enabled=device.type == "cuda",\n    )\n\n    best_checkpoint_path = run_dir / "best_checkpoint.pt"\n\n    best_val_dice = -1.0\n    best_epoch = 0\n    history = []\n\n    for epoch in range(1, MAX_EPOCHS + 1):\n        start = time.time()\n\n        current_lr = optimizer.param_groups[0]["lr"]\n\n        train_loss = train_epoch(\n            train_loader,\n            model,\n            optimizer,\n            scaler,\n            epoch,\n            modality,\n        )\n\n        val_result = validate_model(\n            val_loader,\n            model,\n        )\n\n        scheduler.step(\n            val_result["dice"]\n        )\n\n        row = {\n            "epoch": epoch,\n            "learning_rate": current_lr,\n            "train_loss": train_loss,\n            "val_loss": val_result["loss"],\n            "val_dice": val_result["dice"],\n            "val_iou": val_result["iou"],\n            "val_precision": val_result["precision"],\n            "val_recall": val_result["recall"],\n            "val_threshold_coarse": val_result["threshold"],\n            "minutes": (time.time() - start) / 60.0,\n        }\n\n        history.append(row)\n\n        print(\n            f"{modality} | "\n            f"Epoch {epoch:02d}/{MAX_EPOCHS} | "\n            f"loss={train_loss:.4f} | "\n            f"val Dice={val_result[\'dice\']:.4f} | "\n            f"IoU={val_result[\'iou\']:.4f} | "\n            f"P={val_result[\'precision\']:.4f} | "\n            f"R={val_result[\'recall\']:.4f} | "\n            f"th={val_result[\'threshold\']:.2f} | "\n            f"lr={current_lr:.2e}"\n        )\n\n        if val_result["dice"] > best_val_dice:\n            best_val_dice = val_result["dice"]\n            best_epoch = epoch\n\n            torch.save({\n                "model": model.state_dict(),\n                "modality": modality,\n                "epoch": epoch,\n                "val_dice": best_val_dice,\n                "channels": modality_info(modality)["channels"],\n                "mean": modality_info(modality)["mean"].tolist(),\n                "std": modality_info(modality)["std"].tolist(),\n            }, best_checkpoint_path)\n\n        if epoch - best_epoch >= EARLY_STOPPING_PATIENCE:\n            print(\n                f"Early stopping {modality}: "\n                f"no validation improvement for "\n                f"{EARLY_STOPPING_PATIENCE} epochs."\n            )\n            break\n\n    save_csv(\n        run_dir / "training_history.csv",\n        history,\n    )\n\n    try:\n        checkpoint = torch.load(\n            best_checkpoint_path,\n            map_location=device,\n            weights_only=False,\n        )\n    except TypeError:\n        checkpoint = torch.load(\n            best_checkpoint_path,\n            map_location=device,\n        )\n\n    model.load_state_dict(\n        checkpoint["model"],\n        strict=True,\n    )\n\n    # Fine validation threshold selection\n    val_records = collect_predictions(\n        val_loader,\n        model,\n        f"{modality} validation predictions",\n    )\n\n    val_best, threshold_rows = threshold_search(\n        val_records,\n        FINE_THRESHOLDS,\n    )\n\n    selected_threshold = float(\n        val_best["threshold"]\n    )\n\n    save_csv(\n        run_dir / "validation_threshold_search.csv",\n        threshold_rows,\n    )\n\n    # Untouched official test\n    test_records = collect_predictions(\n        test_loader,\n        model,\n        f"{modality} official test",\n    )\n\n    global_values = [0, 0, 0, 0]\n    per_image_rows = []\n\n    for record in test_records:\n        values = confusion_numpy(\n            record["probability"],\n            record["mask"],\n            selected_threshold,\n        )\n\n        global_values = add_confusion(\n            global_values,\n            values,\n        )\n\n        row = calculate_metrics(values)\n\n        row.update({\n            "stem": record["stem"],\n            "threshold": selected_threshold,\n            "tp": values[0],\n            "fp": values[1],\n            "fn": values[2],\n            "tn": values[3],\n        })\n\n        per_image_rows.append(row)\n\n    test_metrics = calculate_metrics(\n        global_values\n    )\n\n    summary = {\n        "modality": modality,\n        "channels": "+".join(modality_info(modality)["channels"]),\n        "seed": SEED,\n        "train_images": len(train_names),\n        "val_images": len(val_names),\n        "test_images": len(test_names),\n        "parameters": parameter_count,\n        "best_epoch": best_epoch,\n        "selected_threshold": selected_threshold,\n        "dice": test_metrics["dice"],\n        "iou": test_metrics["iou"],\n        "precision": test_metrics["precision"],\n        "recall": test_metrics["recall"],\n        "accuracy": test_metrics["accuracy"],\n        "tp": global_values[0],\n        "fp": global_values[1],\n        "fn": global_values[2],\n        "tn": global_values[3],\n    }\n\n    save_csv(\n        run_dir / "per_image_test_metrics.csv",\n        per_image_rows,\n    )\n\n    save_csv(\n        run_dir / "final_test_metrics.csv",\n        [summary],\n    )\n\n    with open(\n        run_dir / "experiment_configuration.json",\n        "w",\n        encoding="utf-8",\n    ) as f:\n        json.dump({\n            "dataset": "WeedyRice-RGBMS-DB",\n            "protocol": "official train/val/test split",\n            "model": "Plain U-Net",\n            "modality": modality,\n            "channels": modality_info(modality)["channels"],\n            "seed": SEED,\n            "target_size": list(TARGET_SIZE),\n            "max_epochs": MAX_EPOCHS,\n            "early_stopping_patience": EARLY_STOPPING_PATIENCE,\n            "batch_size": BATCH_SIZE,\n            "accumulation_steps": ACCUMULATION_STEPS,\n            "learning_rate": LEARNING_RATE,\n            "weight_decay": WEIGHT_DECAY,\n            "best_epoch": best_epoch,\n            "selected_threshold": selected_threshold,\n            "parameters": parameter_count,\n            "normalization_mean": modality_info(modality)["mean"].tolist(),\n            "normalization_std": modality_info(modality)["std"].tolist(),\n            "final_test_metrics": summary,\n        }, f, indent=2)\n\n    print("\\n" + "-" * 90)\n    print(f"{modality}-ONLY FINAL TEST")\n    print("-" * 90)\n    print("Best epoch :", best_epoch)\n    print("Threshold  :", f"{selected_threshold:.2f}")\n    print("Dice       :", f"{test_metrics[\'dice\']:.4f}")\n    print("IoU        :", f"{test_metrics[\'iou\']:.4f}")\n    print("Precision  :", f"{test_metrics[\'precision\']:.4f}")\n    print("Recall     :", f"{test_metrics[\'recall\']:.4f}")\n    print("Accuracy   :", f"{test_metrics[\'accuracy\']:.4f}")\n\n    # Free GPU memory before next modality\n    del model\n    del optimizer\n    del scaler\n\n    if device.type == "cuda":\n        torch.cuda.empty_cache()\n\n    return summary\n\n\n# =============================================================================\n# 11. RUN BOTH AND SAVE COMPARISON\n# =============================================================================\n\nall_summaries = []\n\nfor modality in MODALITIES:\n    all_summaries.append(\n        run_modality(modality)\n    )\n\nsave_csv(\n    OUT_DIR / "rgb_ms_comparison.csv",\n    all_summaries,\n)\n\nprint("\\n" + "=" * 100)\nprint("RGB-ONLY vs MS-ONLY — OFFICIAL TEST SUMMARY")\nprint("=" * 100)\n\nfor row in all_summaries:\n    print(\n        f"{row[\'modality\']:3s} | "\n        f"Dice={row[\'dice\']:.4f} | "\n        f"IoU={row[\'iou\']:.4f} | "\n        f"Precision={row[\'precision\']:.4f} | "\n        f"Recall={row[\'recall\']:.4f} | "\n        f"Threshold={row[\'selected_threshold\']:.2f} | "\n        f"Epoch={row[\'best_epoch\']}"\n    )\n\narchive_path = shutil.make_archive(\n    str(WORK_DIR / "weedy_rgb_ms_unimodal_seed42_results"),\n    "zip",\n    str(OUT_DIR),\n)\n\nprint("\\nResults directory:", OUT_DIR)\nprint("Results ZIP      :", archive_path)\nprint("\\nDone.")\n'
FUSION_SOURCE = '\n"""\nWeedyRice-RGBMS-DB — Proposed 6-channel gated-fusion model\nFresh training on the official train/val/test split.\n\nInput channels:\n    RGB: R, G, B\n    MS : Green, Red-Edge, NIR\nThe dataset\'s MS Red band is intentionally not used in this first controlled\ntransfer experiment, so the architecture remains exactly 6-channel like the\nGD-RGBMS experiment.\n\nFAST_PILOT=True:\n    max 30 epochs, patience 6\nFAST_PILOT=False:\n    max 70 epochs, patience 12\n"""\n\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport random\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom tqdm import tqdm\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\n\n\n# =============================================================================\n# 1. CONFIG\n# =============================================================================\n\nROOT = Path("/kaggle/input/datasets/hemal7/weedy-rgb-ms")\n\n# Automatically find the actual dataset directory containing the split files.\ntrain_list_candidates = list(ROOT.rglob("train_list.txt"))\nif not train_list_candidates:\n    raise FileNotFoundError(f"Could not find train_list.txt under {ROOT}")\n\nDATA_DIR = train_list_candidates[0].parent\n\nRGB_DIR = DATA_DIR / "RGB"\nMS_DIR = DATA_DIR / "Multispectral"\nMASK_DIR = DATA_DIR / "Masks"\n\nfor p in (RGB_DIR, MS_DIR, MASK_DIR):\n    if not p.exists():\n        raise FileNotFoundError(f"Missing required folder: {p}")\n\nWORK_DIR = Path("/kaggle/working")\nOUT_DIR = WORK_DIR / "weedy_proposed_6ch_seed42"\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nSEED = 42\n\n# Quick first run.\nFAST_PILOT = True\n\nif FAST_PILOT:\n    MAX_EPOCHS = 30\n    EARLY_STOPPING_PATIENCE = 6\nelse:\n    MAX_EPOCHS = 70\n    EARLY_STOPPING_PATIENCE = 12\n\nTARGET_SIZE = (896, 672)  # width, height\nBATCH_SIZE = 2\nACCUMULATION_STEPS = 2\nLEARNING_RATE = 1e-4\nWEIGHT_DECAY = 1e-4\nNUM_WORKERS = 2\n\nBRANCH_BASE = 28\nPREFERRED_GN_GROUPS = 8\nMODALITY_DROPOUT_PROBABILITY = 0.0\n\nRGB_CHANNELS = 3\nSPECTRAL_CHANNELS = 3\nTOTAL_INPUT_CHANNELS = 6\nINPUT_CHANNEL_ORDER = ("R", "G", "B", "Green", "Red-Edge", "NIR")\n\n# Same auxiliary supervision schedule as the GD model.\nAUX_RGB_INITIAL_WEIGHT = 0.15\nAUX_SPECTRAL_INITIAL_WEIGHT = 0.25\nAUX_RGB_MIDDLE_WEIGHT = 0.10\nAUX_SPECTRAL_MIDDLE_WEIGHT = 0.15\nAUX_RGB_FINAL_WEIGHT = 0.05\nAUX_SPECTRAL_FINAL_WEIGHT = 0.08\nAUX_MIDDLE_PHASE_START = 0.40\nAUX_FINAL_PHASE_START = 0.70\n\nCOARSE_THRESHOLDS = np.round(np.arange(0.20, 0.86, 0.05), 2).tolist()\nFINE_THRESHOLDS = np.round(np.arange(0.20, 0.801, 0.01), 2).tolist()\n\nCACHE_DATA = True\n\ntry:\n    BILINEAR = Image.Resampling.BILINEAR\n    NEAREST = Image.Resampling.NEAREST\nexcept AttributeError:\n    BILINEAR = Image.BILINEAR\n    NEAREST = Image.NEAREST\n\n\ndef seed_everything(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\nseed_everything(SEED)\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\npin_memory = device.type == "cuda"\n\nprint("=" * 95)\nprint("WEEDY RICE — PROPOSED 6-CHANNEL GATED-FUSION MODEL")\nprint("=" * 95)\nprint("ROOT        :", ROOT)\nprint("DATA_DIR    :", DATA_DIR)\nprint("RGB_DIR     :", RGB_DIR)\nprint("MS_DIR      :", MS_DIR)\nprint("MASK_DIR    :", MASK_DIR)\nprint("Device      :", device)\nprint("Fast pilot  :", FAST_PILOT)\nprint("Max epochs  :", MAX_EPOCHS)\nprint("Patience    :", EARLY_STOPPING_PATIENCE)\nprint("Input size  :", TARGET_SIZE)\nprint("Channels    :", INPUT_CHANNEL_ORDER)\n\n\n# =============================================================================\n# 2. SPLIT LISTS\n# =============================================================================\n\ndef read_list(path: Path):\n    with open(path, "r", encoding="utf-8") as f:\n        return [line.strip() for line in f if line.strip()]\n\n\ntrain_names = read_list(DATA_DIR / "train_list.txt")\nval_names = read_list(DATA_DIR / "val_list.txt")\ntest_names = read_list(DATA_DIR / "test_list.txt")\n\nprint("\\nOfficial splits:")\nprint("Train:", len(train_names))\nprint("Val  :", len(val_names))\nprint("Test :", len(test_names))\nprint("Total:", len(train_names) + len(val_names) + len(test_names))\n\nif len(set(train_names) & set(val_names)) > 0:\n    raise RuntimeError("Train/val overlap detected.")\nif len(set(train_names) & set(test_names)) > 0:\n    raise RuntimeError("Train/test overlap detected.")\nif len(set(val_names) & set(test_names)) > 0:\n    raise RuntimeError("Val/test overlap detected.")\n\n\n# =============================================================================\n# 3. DATA LOADING\n# =============================================================================\n\ndef filename_stem(name: str) -> str:\n    return Path(name).stem\n\n\ndef image_to_unit_float(path: Path, resize_size=None, resample=BILINEAR):\n    """\n    Reads 8-bit or 16-bit single/multi-channel images and maps integer values\n    linearly to [0,1]. For float images, values are kept if already in [0,1];\n    otherwise they are scaled by their observed max for that image.\n    """\n    with Image.open(path) as img:\n        if resize_size is not None:\n            img = img.resize(resize_size, resample)\n        arr = np.asarray(img).copy()\n\n    original_dtype = arr.dtype\n\n    if np.issubdtype(original_dtype, np.integer):\n        max_value = np.iinfo(original_dtype).max\n        arr = arr.astype(np.float32) / float(max_value)\n    else:\n        arr = arr.astype(np.float32)\n        vmax = float(np.nanmax(arr)) if arr.size else 1.0\n        if vmax > 1.0:\n            arr = arr / max(vmax, 1e-6)\n\n    return arr\n\n\ndef load_rgb(path: Path):\n    with Image.open(path) as img:\n        img = img.convert("RGB").resize(TARGET_SIZE, BILINEAR)\n        rgb = np.asarray(img, dtype=np.float32).copy() / 255.0\n    return rgb\n\n\ndef load_single_band(path: Path):\n    arr = image_to_unit_float(path, resize_size=TARGET_SIZE, resample=BILINEAR)\n\n    if arr.ndim == 3:\n        arr = arr[..., 0]\n\n    if arr.ndim != 2:\n        raise RuntimeError(f"Expected a single-band image, got {arr.shape}: {path}")\n\n    return arr.astype(np.float32)\n\n\ndef load_mask(path: Path):\n    with Image.open(path) as img:\n        img = img.convert("L").resize(TARGET_SIZE, NEAREST)\n        arr = np.asarray(img, dtype=np.uint8)\n    # Official masks: 255 = weedy rice, 0 = background\n    return (arr > 127).astype(np.float32)\n\n\ndef paths_for(name: str):\n    stem = filename_stem(name)\n\n    rgb_path = RGB_DIR / f"{stem}.JPG"\n    if not rgb_path.exists():\n        rgb_path = RGB_DIR / f"{stem}.jpg"\n\n    g_path = MS_DIR / f"{stem}_G.TIF"\n    re_path = MS_DIR / f"{stem}_RE.TIF"\n    nir_path = MS_DIR / f"{stem}_NIR.TIF"\n\n    mask_path = MASK_DIR / f"{stem}.png"\n\n    for p in (rgb_path, g_path, re_path, nir_path, mask_path):\n        if not p.exists():\n            raise FileNotFoundError(f"Missing paired file: {p}")\n\n    return rgb_path, g_path, re_path, nir_path, mask_path\n\n\ndef load_raw(name: str):\n    rgb_path, g_path, re_path, nir_path, mask_path = paths_for(name)\n\n    rgb = load_rgb(rgb_path)\n    green = load_single_band(g_path)\n    red_edge = load_single_band(re_path)\n    nir = load_single_band(nir_path)\n    mask = load_mask(mask_path)\n\n    ms = np.stack([green, red_edge, nir], axis=-1)\n    image = np.concatenate([rgb, ms], axis=-1).astype(np.float32)\n\n    if image.shape != (TARGET_SIZE[1], TARGET_SIZE[0], 6):\n        raise RuntimeError(f"Unexpected image shape {image.shape} for {name}")\n\n    return image, mask, filename_stem(name)\n\n\n# Quick pairing check.\nprint("\\nChecking one sample...")\nsample_image, sample_mask, sample_stem = load_raw(train_names[0])\nprint("Sample stem :", sample_stem)\nprint("Image shape :", sample_image.shape)\nprint("Mask shape  :", sample_mask.shape)\nprint("Image range :", float(sample_image.min()), "to", float(sample_image.max()))\nprint("Mask values :", np.unique(sample_mask))\n\n\n# =============================================================================\n# 4. OPTIONAL RAM CACHE\n# =============================================================================\n\nRAW_CACHE = {}\n\nall_names = train_names + val_names + test_names\n\nif CACHE_DATA:\n    print("\\nCaching resized 6-channel samples...")\n    for name in tqdm(all_names, desc="Cache"):\n        image, mask, stem = load_raw(name)\n        RAW_CACHE[name] = (\n            image.astype(np.float16),\n            mask.astype(np.uint8),\n            stem,\n        )\n    total_gb = sum(\n        x.nbytes + y.nbytes for x, y, _ in RAW_CACHE.values()\n    ) / (1024 ** 3)\n    print(f"Cached {len(RAW_CACHE)} samples (~{total_gb:.2f} GB)")\n\n\ndef get_raw(name):\n    if name in RAW_CACHE:\n        image, mask, stem = RAW_CACHE[name]\n        return image.astype(np.float32), mask.astype(np.float32), stem\n    return load_raw(name)\n\n\n# =============================================================================\n# 5. TRAIN-ONLY NORMALIZATION\n# =============================================================================\n\ndef compute_train_stats(names):\n    channel_sum = np.zeros(TOTAL_INPUT_CHANNELS, dtype=np.float64)\n    channel_sq = np.zeros(TOTAL_INPUT_CHANNELS, dtype=np.float64)\n    count = 0\n\n    for name in tqdm(names, desc="Train normalization"):\n        image, _, _ = get_raw(name)\n\n        channel_sum += image.sum(axis=(0, 1), dtype=np.float64)\n        channel_sq += np.square(image, dtype=np.float32).sum(\n            axis=(0, 1), dtype=np.float64\n        )\n        count += image.shape[0] * image.shape[1]\n\n    mean64 = channel_sum / count\n    var64 = channel_sq / count - np.square(mean64)\n\n    mean = mean64.astype(np.float32)\n    std = np.sqrt(np.maximum(var64, 1e-12)).astype(np.float32)\n\n    return mean, std\n\n\nMEAN, STD = compute_train_stats(train_names)\n\nprint("\\nTraining-set normalization:")\nfor name, mean, std in zip(INPUT_CHANNEL_ORDER, MEAN, STD):\n    print(f"{name:10s} mean={mean:.6f} std={std:.6f}")\n\nwith open(OUT_DIR / "normalization.json", "w", encoding="utf-8") as f:\n    json.dump({\n        "channel_order": list(INPUT_CHANNEL_ORDER),\n        "mean": MEAN.tolist(),\n        "std": STD.tolist(),\n        "source": "official training split only",\n    }, f, indent=2)\n\n\n# =============================================================================\n# 6. DATASET CLASS\n# =============================================================================\n\nclass WeedyRiceDataset(Dataset):\n    def __init__(self, names, mean, std, augment=False):\n        self.names = list(names)\n        self.mean = np.asarray(mean, dtype=np.float32).reshape(1, 1, -1)\n        self.std = np.asarray(std, dtype=np.float32).reshape(1, 1, -1)\n        self.augment = augment\n\n    def __len__(self):\n        return len(self.names)\n\n    def __getitem__(self, index):\n        name = self.names[index]\n        image, mask, stem = get_raw(name)\n\n        if self.augment:\n            if random.random() < 0.5:\n                image = np.flip(image, axis=1).copy()\n                mask = np.flip(mask, axis=1).copy()\n\n            if random.random() < 0.5:\n                image = np.flip(image, axis=0).copy()\n                mask = np.flip(mask, axis=0).copy()\n\n            # Same visible-image perturbation used for GD.\n            if random.random() < 0.75:\n                gain = np.random.uniform(\n                    0.88, 1.12, size=(1, 1, 3)\n                ).astype(np.float32)\n                bias = np.random.uniform(\n                    -0.035, 0.035, size=(1, 1, 3)\n                ).astype(np.float32)\n                image[..., :3] = np.clip(\n                    image[..., :3] * gain + bias, 0.0, 1.0\n                )\n\n        image = ((image - self.mean) / self.std).astype(np.float32)\n\n        return (\n            torch.from_numpy(image.transpose(2, 0, 1)).float(),\n            torch.from_numpy(mask[None]).float(),\n            stem,\n        )\n\n\ndef make_loader(names, augment=False, batch_size=BATCH_SIZE, shuffle=False):\n    return DataLoader(\n        WeedyRiceDataset(names, MEAN, STD, augment=augment),\n        batch_size=batch_size,\n        shuffle=shuffle,\n        num_workers=NUM_WORKERS,\n        pin_memory=pin_memory,\n        drop_last=False,\n    )\n\n\ntrain_loader = make_loader(train_names, augment=True, batch_size=BATCH_SIZE, shuffle=True)\nval_loader = make_loader(val_names, augment=False, batch_size=1, shuffle=False)\ntest_loader = make_loader(test_names, augment=False, batch_size=1, shuffle=False)\n\n\n# =============================================================================\n# 7. EXACT PROPOSED MODEL\n# =============================================================================\n\ndef group_norm(channels):\n    groups = min(PREFERRED_GN_GROUPS, channels)\n    while channels % groups:\n        groups -= 1\n    return nn.GroupNorm(groups, channels, eps=1e-5, affine=True)\n\n\nclass SCSE(nn.Module):\n    def __init__(self, channels, reduction=16):\n        super().__init__()\n        hidden = max(channels // reduction, 1)\n\n        self.channel = nn.Sequential(\n            nn.AdaptiveAvgPool2d(1),\n            nn.Conv2d(channels, hidden, 1),\n            nn.SiLU(inplace=True),\n            nn.Conv2d(hidden, channels, 1),\n            nn.Sigmoid(),\n        )\n\n        self.spatial = nn.Sequential(\n            nn.Conv2d(channels, 1, 1),\n            nn.Sigmoid(),\n        )\n\n    def forward(self, x):\n        return x * self.channel(x) + x * self.spatial(x)\n\n\nclass ResBlock(nn.Module):\n    def __init__(self, in_channels, out_channels, dropout=0.0):\n        super().__init__()\n\n        self.block = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),\n            group_norm(out_channels),\n            nn.SiLU(inplace=True),\n            nn.Dropout2d(dropout) if dropout else nn.Identity(),\n            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),\n            group_norm(out_channels),\n        )\n\n        self.skip = (\n            nn.Identity()\n            if in_channels == out_channels\n            else nn.Sequential(\n                nn.Conv2d(in_channels, out_channels, 1, bias=False),\n                group_norm(out_channels),\n            )\n        )\n\n        self.attention = SCSE(out_channels)\n\n    def forward(self, x):\n        y = self.block(x) + self.skip(x)\n        return self.attention(F.silu(y, inplace=True))\n\n\nclass SoftGatedFusion(nn.Module):\n    def __init__(self, channels):\n        super().__init__()\n        hidden = max(channels // 8, 8)\n\n        self.gate = nn.Sequential(\n            nn.Conv2d(channels * 2, hidden, 1, bias=False),\n            group_norm(hidden),\n            nn.SiLU(inplace=True),\n            nn.Conv2d(hidden, 2, 1),\n        )\n\n        self.refine = nn.Sequential(\n            nn.Conv2d(channels, channels, 3, padding=1, bias=False),\n            group_norm(channels),\n            nn.SiLU(inplace=True),\n        )\n\n    def forward(self, rgb_feature, spectral_feature):\n        weights = torch.softmax(\n            self.gate(torch.cat([rgb_feature, spectral_feature], dim=1)),\n            dim=1,\n        )\n\n        fused = (\n            weights[:, 0:1] * rgb_feature\n            + weights[:, 1:2] * spectral_feature\n        )\n\n        return self.refine(fused), weights\n\n\nclass UpBlock(nn.Module):\n    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):\n        super().__init__()\n        self.conv = ResBlock(\n            in_channels + skip_channels,\n            out_channels,\n            dropout,\n        )\n\n    def forward(self, x, skip):\n        x = F.interpolate(\n            x,\n            size=skip.shape[-2:],\n            mode="bilinear",\n            align_corners=False,\n        )\n        return self.conv(torch.cat([x, skip], dim=1))\n\n\nclass AuxiliarySupervisedGatedUNetNoNDRE(nn.Module):\n    def __init__(self, base=28):\n        super().__init__()\n\n        c1, c2, c3, c4 = [\n            base,\n            base * 2,\n            base * 4,\n            base * 8,\n        ]\n\n        self.pool = nn.MaxPool2d(2)\n\n        self.rgb1 = ResBlock(RGB_CHANNELS, c1)\n        self.spec1 = ResBlock(SPECTRAL_CHANNELS, c1)\n\n        self.rgb2 = ResBlock(c1, c2)\n        self.spec2 = ResBlock(c1, c2)\n\n        self.rgb3 = ResBlock(c2, c3, 0.05)\n        self.spec3 = ResBlock(c2, c3, 0.05)\n\n        self.rgb4 = ResBlock(c3, c4, 0.10)\n        self.spec4 = ResBlock(c3, c4, 0.10)\n\n        self.fuse1 = SoftGatedFusion(c1)\n        self.fuse2 = SoftGatedFusion(c2)\n        self.fuse3 = SoftGatedFusion(c3)\n        self.fuse4 = SoftGatedFusion(c4)\n\n        self.rgb_auxiliary_head = nn.Conv2d(c4, 1, 1)\n        self.spectral_auxiliary_head = nn.Conv2d(c4, 1, 1)\n\n        self.center = ResBlock(c4, c4 * 2, 0.10)\n\n        self.up4 = UpBlock(c4 * 2, c4, c4, 0.10)\n        self.up3 = UpBlock(c4, c3, c3, 0.05)\n        self.up2 = UpBlock(c3, c2, c2)\n        self.up1 = UpBlock(c2, c1, c1)\n\n        self.output = nn.Conv2d(c1, 1, 1)\n\n    def modality_dropout(self, rgb, spectral):\n        if (\n            not self.training\n            or MODALITY_DROPOUT_PROBABILITY <= 0\n        ):\n            return rgb, spectral\n\n        draw = random.random()\n\n        if draw < MODALITY_DROPOUT_PROBABILITY / 2:\n            rgb = torch.zeros_like(rgb)\n        elif draw < MODALITY_DROPOUT_PROBABILITY:\n            spectral = torch.zeros_like(spectral)\n\n        return rgb, spectral\n\n    def forward(self, x, return_aux=False):\n        if x.ndim != 4 or x.shape[1] != 6:\n            raise ValueError(\n                f"Expected [B,6,H,W], got {tuple(x.shape)}"\n            )\n\n        input_size = x.shape[-2:]\n\n        rgb, spectral = self.modality_dropout(\n            x[:, :3],\n            x[:, 3:],\n        )\n\n        r1 = self.rgb1(rgb)\n        s1 = self.spec1(spectral)\n        f1, _ = self.fuse1(r1, s1)\n\n        r2 = self.rgb2(self.pool(r1 + f1))\n        s2 = self.spec2(self.pool(s1 + f1))\n        f2, _ = self.fuse2(r2, s2)\n\n        r3 = self.rgb3(self.pool(r2 + f2))\n        s3 = self.spec3(self.pool(s2 + f2))\n        f3, _ = self.fuse3(r3, s3)\n\n        r4 = self.rgb4(self.pool(r3 + f3))\n        s4 = self.spec4(self.pool(s3 + f3))\n        f4, _ = self.fuse4(r4, s4)\n\n        center = self.center(self.pool(f4))\n\n        d4 = self.up4(center, f4)\n        d3 = self.up3(d4, f3)\n        d2 = self.up2(d3, f2)\n        d1 = self.up1(d2, f1)\n\n        logits = self.output(d1)\n\n        if return_aux:\n            rgb_aux = F.interpolate(\n                self.rgb_auxiliary_head(r4),\n                size=input_size,\n                mode="bilinear",\n                align_corners=False,\n            )\n\n            spec_aux = F.interpolate(\n                self.spectral_auxiliary_head(s4),\n                size=input_size,\n                mode="bilinear",\n                align_corners=False,\n            )\n\n            return logits, rgb_aux, spec_aux\n\n        return logits\n\n\ndef build_model():\n    model = AuxiliarySupervisedGatedUNetNoNDRE(BRANCH_BASE).to(device)\n\n    total = sum(p.numel() for p in model.parameters())\n    aux = sum(\n        p.numel()\n        for name, p in model.named_parameters()\n        if "auxiliary_head" in name\n    )\n    deploy = total - aux\n\n    print(\n        f"Parameters: total={total:,}, "\n        f"auxiliary={aux:,}, deployment={deploy:,}"\n    )\n\n    return model, total, aux, deploy\n\n\n# =============================================================================\n# 8. LOSS / METRICS\n# =============================================================================\n\nclass ComboLoss(nn.Module):\n    def forward(self, logits, targets):\n        bce_map = F.binary_cross_entropy_with_logits(\n            logits,\n            targets.float(),\n            reduction="none",\n        )\n        bce = bce_map.mean()\n\n        probabilities = torch.sigmoid(logits)\n\n        pt = (\n            probabilities * targets\n            + (1 - probabilities) * (1 - targets)\n        )\n        focal = (\n            bce_map * ((1 - pt) ** 2.0)\n        ).mean()\n\n        dims = tuple(range(1, logits.ndim))\n\n        intersection = (\n            probabilities * targets\n        ).sum(dims)\n\n        denominator = (\n            probabilities.sum(dims)\n            + targets.sum(dims)\n        )\n\n        dice = 1 - (\n            (2 * intersection + 1e-6)\n            / (denominator + 1e-6)\n        ).mean()\n\n        return (\n            0.35 * bce\n            + 0.25 * focal\n            + 0.40 * dice\n        )\n\n\ncriterion = ComboLoss()\n\n\ndef auxiliary_loss_weights(epoch):\n    progress = epoch / max(MAX_EPOCHS, 1)\n\n    if progress <= AUX_MIDDLE_PHASE_START:\n        return (\n            AUX_RGB_INITIAL_WEIGHT,\n            AUX_SPECTRAL_INITIAL_WEIGHT,\n        )\n\n    if progress <= AUX_FINAL_PHASE_START:\n        return (\n            AUX_RGB_MIDDLE_WEIGHT,\n            AUX_SPECTRAL_MIDDLE_WEIGHT,\n        )\n\n    return (\n        AUX_RGB_FINAL_WEIGHT,\n        AUX_SPECTRAL_FINAL_WEIGHT,\n    )\n\n\ndef confusion(probabilities, masks, threshold):\n    if isinstance(probabilities, np.ndarray):\n        prediction = probabilities >= threshold\n        target = masks >= 0.5\n\n        tp = int(np.logical_and(prediction, target).sum())\n        fp = int(np.logical_and(prediction, ~target).sum())\n        fn = int(np.logical_and(~prediction, target).sum())\n        tn = int(np.logical_and(~prediction, ~target).sum())\n\n        return [tp, fp, fn, tn]\n\n    prediction = probabilities >= threshold\n    target = masks >= 0.5\n\n    tp = int((prediction & target).sum().item())\n    fp = int((prediction & ~target).sum().item())\n    fn = int((~prediction & target).sum().item())\n    tn = int((~prediction & ~target).sum().item())\n\n    return [tp, fp, fn, tn]\n\n\ndef metrics(values):\n    tp, fp, fn, tn = values\n    eps = 1e-7\n\n    return {\n        "dice": (2 * tp + eps) / (2 * tp + fp + fn + eps),\n        "iou": (tp + eps) / (tp + fp + fn + eps),\n        "precision": (tp + eps) / (tp + fp + eps),\n        "recall": (tp + eps) / (tp + fn + eps),\n        "accuracy": (tp + tn + eps) / (tp + fp + fn + tn + eps),\n    }\n\n\ndef save_csv(path, rows):\n    if not rows:\n        return\n\n    with open(path, "w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(\n            f,\n            fieldnames=list(rows[0].keys()),\n        )\n        writer.writeheader()\n        writer.writerows(rows)\n\n\n# =============================================================================\n# 9. TRAIN / VALIDATE\n# =============================================================================\n\ndef train_epoch(loader, epoch, model, optimizer, scaler):\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n\n    rgb_weight, spectral_weight = auxiliary_loss_weights(epoch)\n\n    sums = {\n        "total": 0.0,\n        "main": 0.0,\n        "rgb_aux": 0.0,\n        "spec_aux": 0.0,\n    }\n\n    for step, (images, masks, _) in enumerate(\n        tqdm(loader, desc=f"Train epoch {epoch}", leave=False),\n        start=1,\n    ):\n        images = images.to(device, non_blocking=True)\n        masks = masks.to(device, non_blocking=True)\n\n        with torch.amp.autocast(\n            "cuda",\n            enabled=device.type == "cuda",\n        ):\n            main_logits, rgb_aux_logits, spec_aux_logits = model(\n                images,\n                return_aux=True,\n            )\n\n            main_loss = criterion(main_logits, masks)\n            rgb_aux_loss = criterion(rgb_aux_logits, masks)\n            spec_aux_loss = criterion(spec_aux_logits, masks)\n\n            total_loss = (\n                main_loss\n                + rgb_weight * rgb_aux_loss\n                + spectral_weight * spec_aux_loss\n            )\n\n            scaled_loss = (\n                total_loss / ACCUMULATION_STEPS\n            )\n\n        scaler.scale(scaled_loss).backward()\n\n        sums["total"] += total_loss.item()\n        sums["main"] += main_loss.item()\n        sums["rgb_aux"] += rgb_aux_loss.item()\n        sums["spec_aux"] += spec_aux_loss.item()\n\n        if (\n            step % ACCUMULATION_STEPS == 0\n            or step == len(loader)\n        ):\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(\n                model.parameters(),\n                1.0,\n            )\n\n            scaler.step(optimizer)\n            scaler.update()\n            optimizer.zero_grad(set_to_none=True)\n\n    return {\n        key: value / len(loader)\n        for key, value in sums.items()\n    }\n\n\n@torch.no_grad()\ndef validate_model(loader, model):\n    model.eval()\n\n    totals = {\n        t: [0, 0, 0, 0]\n        for t in COARSE_THRESHOLDS\n    }\n\n    loss_total = 0.0\n\n    for images, masks, _ in tqdm(\n        loader,\n        desc="Validation",\n        leave=False,\n    ):\n        images = images.to(device, non_blocking=True)\n        masks = masks.to(device, non_blocking=True)\n\n        with torch.amp.autocast(\n            "cuda",\n            enabled=device.type == "cuda",\n        ):\n            logits = model(images)\n            loss_total += criterion(logits, masks).item()\n\n        probabilities = torch.sigmoid(logits)\n\n        for threshold in COARSE_THRESHOLDS:\n            values = confusion(\n                probabilities,\n                masks,\n                threshold,\n            )\n            totals[threshold] = [\n                a + b\n                for a, b in zip(\n                    totals[threshold],\n                    values,\n                )\n            ]\n\n    rows = []\n\n    for threshold, values in totals.items():\n        row = metrics(values)\n        row["threshold"] = float(threshold)\n        row["loss"] = loss_total / len(loader)\n        rows.append(row)\n\n    return max(rows, key=lambda row: row["dice"])\n\n\n@torch.no_grad()\ndef collect_probabilities(loader, model, description):\n    model.eval()\n    records = []\n\n    for images, masks, stems in tqdm(\n        loader,\n        desc=description,\n        leave=False,\n    ):\n        images = images.to(device, non_blocking=True)\n\n        with torch.amp.autocast(\n            "cuda",\n            enabled=device.type == "cuda",\n        ):\n            probabilities = torch.sigmoid(\n                model(images)\n            )\n\n        records.append({\n            "stem": stems[0],\n            "mask": masks[0, 0].numpy().astype(np.float32),\n            "probability": probabilities[\n                0, 0\n            ].cpu().numpy().astype(np.float32),\n        })\n\n    return records\n\n\ndef threshold_search(records, thresholds):\n    rows = []\n\n    for threshold in thresholds:\n        total = [0, 0, 0, 0]\n\n        for record in records:\n            values = confusion(\n                record["probability"],\n                record["mask"],\n                threshold,\n            )\n\n            total = [\n                a + b\n                for a, b in zip(total, values)\n            ]\n\n        row = metrics(total)\n        row["threshold"] = float(threshold)\n        rows.append(row)\n\n    return max(\n        rows,\n        key=lambda row: row["dice"],\n    ), rows\n\n\n# =============================================================================\n# 10. TRAIN MODEL\n# =============================================================================\n\nseed_everything(SEED)\n\nmodel, parameter_count, auxiliary_parameter_count, deployment_parameter_count = build_model()\n\noptimizer = torch.optim.AdamW(\n    model.parameters(),\n    lr=LEARNING_RATE,\n    weight_decay=WEIGHT_DECAY,\n)\n\nscheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(\n    optimizer,\n    mode="max",\n    factor=0.5,\n    patience=4,\n    min_lr=1e-6,\n)\n\nscaler = torch.amp.GradScaler(\n    "cuda",\n    enabled=device.type == "cuda",\n)\n\nbest_checkpoint_path = OUT_DIR / "best_checkpoint.pt"\n\nhistory = []\n\nbest_val_dice = -1.0\nbest_epoch = 0\n\nprint("\\n" + "=" * 95)\nprint("TRAINING")\nprint("=" * 95)\n\nfor epoch in range(1, MAX_EPOCHS + 1):\n    start = time.time()\n\n    lr_used = optimizer.param_groups[0]["lr"]\n\n    train_result = train_epoch(\n        train_loader,\n        epoch,\n        model,\n        optimizer,\n        scaler,\n    )\n\n    val_result = validate_model(\n        val_loader,\n        model,\n    )\n\n    scheduler.step(val_result["dice"])\n\n    row = {\n        "epoch": epoch,\n        "learning_rate": lr_used,\n        "train_total_loss": train_result["total"],\n        "train_main_loss": train_result["main"],\n        "train_rgb_aux_loss": train_result["rgb_aux"],\n        "train_spec_aux_loss": train_result["spec_aux"],\n        "val_loss": val_result["loss"],\n        "val_dice": val_result["dice"],\n        "val_iou": val_result["iou"],\n        "val_precision": val_result["precision"],\n        "val_recall": val_result["recall"],\n        "val_threshold_coarse": val_result["threshold"],\n        "minutes": (time.time() - start) / 60.0,\n    }\n\n    history.append(row)\n\n    print(\n        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "\n        f"loss={train_result[\'total\']:.4f} | "\n        f"val Dice={val_result[\'dice\']:.4f} | "\n        f"IoU={val_result[\'iou\']:.4f} | "\n        f"P={val_result[\'precision\']:.4f} | "\n        f"R={val_result[\'recall\']:.4f} | "\n        f"th={val_result[\'threshold\']:.2f} | "\n        f"lr={lr_used:.2e}"\n    )\n\n    if val_result["dice"] > best_val_dice:\n        best_val_dice = val_result["dice"]\n        best_epoch = epoch\n\n        torch.save({\n            "model": model.state_dict(),\n            "epoch": epoch,\n            "val_dice": best_val_dice,\n            "mean": MEAN.tolist(),\n            "std": STD.tolist(),\n            "channel_order": list(INPUT_CHANNEL_ORDER),\n        }, best_checkpoint_path)\n\n    if epoch - best_epoch >= EARLY_STOPPING_PATIENCE:\n        print(\n            f"\\nEarly stopping: no validation improvement "\n            f"for {EARLY_STOPPING_PATIENCE} epochs."\n        )\n        break\n\nsave_csv(\n    OUT_DIR / "training_history.csv",\n    history,\n)\n\n\n# =============================================================================\n# 11. FINE THRESHOLD SEARCH ON OFFICIAL VALIDATION SET\n# =============================================================================\n\ntry:\n    checkpoint = torch.load(\n        best_checkpoint_path,\n        map_location=device,\n        weights_only=False,\n    )\nexcept TypeError:\n    checkpoint = torch.load(\n        best_checkpoint_path,\n        map_location=device,\n    )\n\nmodel.load_state_dict(\n    checkpoint["model"],\n    strict=True,\n)\n\nval_records = collect_probabilities(\n    val_loader,\n    model,\n    "Fine threshold search on validation",\n)\n\nval_best, threshold_rows = threshold_search(\n    val_records,\n    FINE_THRESHOLDS,\n)\n\nselected_threshold = float(\n    val_best["threshold"]\n)\n\nsave_csv(\n    OUT_DIR / "validation_threshold_search.csv",\n    threshold_rows,\n)\n\nprint("\\n" + "=" * 95)\nprint("VALIDATION SELECTION")\nprint("=" * 95)\nprint("Best epoch        :", best_epoch)\nprint("Best val Dice     :", f"{best_val_dice:.4f}")\nprint("Selected threshold:", f"{selected_threshold:.2f}")\nprint("Fine val Dice     :", f"{val_best[\'dice\']:.4f}")\n\n\n# =============================================================================\n# 12. FINAL TEST — OFFICIAL TEST SET, UNTOUCHED\n# =============================================================================\n\ntest_records = collect_probabilities(\n    test_loader,\n    model,\n    "Official test evaluation",\n)\n\nglobal_values = [0, 0, 0, 0]\nper_image_rows = []\n\nfor record in test_records:\n    values = confusion(\n        record["probability"],\n        record["mask"],\n        selected_threshold,\n    )\n\n    global_values = [\n        a + b\n        for a, b in zip(\n            global_values,\n            values,\n        )\n    ]\n\n    row = metrics(values)\n\n    row.update({\n        "stem": record["stem"],\n        "threshold": selected_threshold,\n        "tp": values[0],\n        "fp": values[1],\n        "fn": values[2],\n        "tn": values[3],\n    })\n\n    per_image_rows.append(row)\n\ntest_metrics = metrics(global_values)\n\ntest_metrics.update({\n    "seed": SEED,\n    "train_images": len(train_names),\n    "val_images": len(val_names),\n    "test_images": len(test_names),\n    "best_epoch": best_epoch,\n    "selected_threshold": selected_threshold,\n    "fast_pilot": FAST_PILOT,\n    "tp": global_values[0],\n    "fp": global_values[1],\n    "fn": global_values[2],\n    "tn": global_values[3],\n})\n\nsave_csv(\n    OUT_DIR / "per_image_test_metrics.csv",\n    per_image_rows,\n)\n\nsave_csv(\n    OUT_DIR / "final_test_metrics.csv",\n    [test_metrics],\n)\n\nwith open(\n    OUT_DIR / "experiment_configuration.json",\n    "w",\n    encoding="utf-8",\n) as f:\n    json.dump({\n        "dataset": "WeedyRice-RGBMS-DB",\n        "protocol": "official train/val/test split",\n        "seed": SEED,\n        "splits": {\n            "train": len(train_names),\n            "val": len(val_names),\n            "test": len(test_names),\n        },\n        "input_channel_order": list(INPUT_CHANNEL_ORDER),\n        "unused_ms_band": "Red",\n        "target_size": list(TARGET_SIZE),\n        "batch_size": BATCH_SIZE,\n        "accumulation_steps": ACCUMULATION_STEPS,\n        "max_epochs": MAX_EPOCHS,\n        "early_stopping_patience": EARLY_STOPPING_PATIENCE,\n        "learning_rate": LEARNING_RATE,\n        "weight_decay": WEIGHT_DECAY,\n        "best_epoch": best_epoch,\n        "selected_threshold": selected_threshold,\n        "normalization_mean": MEAN.tolist(),\n        "normalization_std": STD.tolist(),\n        "parameter_count": parameter_count,\n        "deployment_parameter_count": deployment_parameter_count,\n        "final_test_metrics": test_metrics,\n    }, f, indent=2)\n\n\n# =============================================================================\n# 13. SUMMARY / ZIP\n# =============================================================================\n\nprint("\\n" + "=" * 95)\nprint("FINAL OFFICIAL TEST RESULT")\nprint("=" * 95)\nprint("Train      :", len(train_names))\nprint("Validation :", len(val_names))\nprint("Test       :", len(test_names))\nprint("Best epoch :", best_epoch)\nprint("Threshold  :", f"{selected_threshold:.2f}")\nprint("Dice       :", f"{test_metrics[\'dice\']:.4f}")\nprint("IoU        :", f"{test_metrics[\'iou\']:.4f}")\nprint("Precision  :", f"{test_metrics[\'precision\']:.4f}")\nprint("Recall     :", f"{test_metrics[\'recall\']:.4f}")\nprint("Accuracy   :", f"{test_metrics[\'accuracy\']:.4f}")\n\narchive_path = shutil.make_archive(\n    str(WORK_DIR / "weedy_proposed_6ch_seed42_results"),\n    "zip",\n    str(OUT_DIR),\n)\n\nprint("\\nResults directory:", OUT_DIR)\nprint("Results ZIP      :", archive_path)\nprint("\\nDone.")\n'


def disable_tqdm(text):
    # Replace either tqdm.auto or tqdm import with a wrapper that disables all bars.
    text = re.sub(
        r"from tqdm(?:\.auto)? import tqdm",
        """from tqdm import tqdm as _tqdm_original

def tqdm(*args, **kwargs):
    kwargs["disable"] = True
    return _tqdm_original(*args, **kwargs)
""",
        text,
        count=1,
    )
    return text


def prepare_baseline(seed):
    text = BASELINE_SOURCE

    text = re.sub(
        r"(?m)^SEED\s*=\s*\d+\s*$",
        f"SEED = {seed}",
        text,
        count=1,
    )

    # Match your completed seed-42 unimodal run: 50 epochs, patience 12.
    text = re.sub(
        r"(?m)^MAX_EPOCHS\s*=\s*\d+\s*$",
        "MAX_EPOCHS = 50",
        text,
        count=1,
    )
    text = re.sub(
        r"(?m)^EARLY_STOPPING_PATIENCE\s*=\s*\d+\s*$",
        "EARLY_STOPPING_PATIENCE = 12",
        text,
        count=1,
    )

    text = re.sub(
        r"(?m)^NUM_WORKERS\s*=\s*\d+\s*$",
        "NUM_WORKERS = 0",
        text,
        count=1,
    )

    text = text.replace(
        'OUT_DIR = WORK_DIR / "weedy_rgb_ms_unimodal_seed42"',
        f'OUT_DIR = WORK_DIR / "weedy_rgb_ms_unimodal_seed{seed}"',
    )
    text = text.replace(
        'weedy_rgb_ms_unimodal_seed42_results',
        f'weedy_rgb_ms_unimodal_seed{seed}_results',
    )

    text = disable_tqdm(text)

    temp = WORK / f"_run_unimodal_seed{seed}.py"
    temp.write_text(text, encoding="utf-8")
    compile(text, str(temp), "exec")
    return temp


def prepare_fusion(seed):
    text = FUSION_SOURCE

    text = re.sub(
        r"(?m)^SEED\s*=\s*\d+\s*$",
        f"SEED = {seed}",
        text,
        count=1,
    )

    # Full proposed-model training, matching the full seed-42 run.
    text = re.sub(
        r"(?m)^FAST_PILOT\s*=\s*(True|False)\s*$",
        "FAST_PILOT = False",
        text,
        count=1,
    )

    text = re.sub(
        r"(?m)^NUM_WORKERS\s*=\s*\d+\s*$",
        "NUM_WORKERS = 0",
        text,
        count=1,
    )

    text = text.replace(
        'OUT_DIR = WORK_DIR / "weedy_proposed_6ch_seed42"',
        f'OUT_DIR = WORK_DIR / "weedy_proposed_6ch_seed{seed}"',
    )
    text = text.replace(
        'weedy_proposed_6ch_seed42_results',
        f'weedy_proposed_6ch_seed{seed}_results',
    )

    text = disable_tqdm(text)

    temp = WORK / f"_run_fusion_seed{seed}.py"
    temp.write_text(text, encoding="utf-8")
    compile(text, str(temp), "exec")
    return temp


def run_file(path, title):
    print("\n" + "=" * 100, flush=True)
    print(title, flush=True)
    print("=" * 100, flush=True)

    subprocess.run(
        [sys.executable, "-u", str(path)],
        check=True,
    )

    path.unlink(missing_ok=True)


print("=" * 100)
print("WEEDY RICE: SEEDS 123 AND 2026 ONLY")
print("=" * 100)
print("Runs:")
print("  Seed 123  : RGB-only + MS-only + Proposed RGB+MS")
print("  Seed 2026 : RGB-only + MS-only + Proposed RGB+MS")
print("No seed-42 run will be started.")
print("No previous model/checkpoint is required.")


for seed in SEEDS:
    # One baseline script trains RGB-only and then MS-only.
    baseline_file = prepare_baseline(seed)
    run_file(
        baseline_file,
        f"SEED {seed} — RGB-ONLY + MS-ONLY",
    )

    # Proposed fusion model.
    fusion_file = prepare_fusion(seed)
    run_file(
        fusion_file,
        f"SEED {seed} — PROPOSED RGB+MS",
    )


# -------------------------------------------------------------------------
# Collect only the newly run seed 123 and 2026 results.
# NO mean/SD calculation.
# -------------------------------------------------------------------------
rows = []

for seed in SEEDS:
    baseline_root = WORK / f"weedy_rgb_ms_unimodal_seed{seed}"

    for modality, label in [
        ("rgb", "RGB-only U-Net"),
        ("ms", "MS-only U-Net"),
    ]:
        p = baseline_root / modality / "final_test_metrics.csv"
        if p.exists():
            d = pd.read_csv(p).iloc[0]
            rows.append({
                "Model": label,
                "Seed": seed,
                "Dice": float(d["dice"]),
                "IoU": float(d["iou"]),
                "Precision": float(d["precision"]),
                "Recall": float(d["recall"]),
                "Accuracy": float(d["accuracy"]),
                "Threshold": float(d["selected_threshold"]),
                "Best_epoch": int(d["best_epoch"]),
            })

    p = (
        WORK
        / f"weedy_proposed_6ch_seed{seed}"
        / "final_test_metrics.csv"
    )

    if p.exists():
        d = pd.read_csv(p).iloc[0]
        rows.append({
            "Model": "Proposed RGB+MS",
            "Seed": seed,
            "Dice": float(d["dice"]),
            "IoU": float(d["iou"]),
            "Precision": float(d["precision"]),
            "Recall": float(d["recall"]),
            "Accuracy": float(d["accuracy"]),
            "Threshold": float(d["selected_threshold"]),
            "Best_epoch": int(d["best_epoch"]),
        })


if rows:
    results = pd.DataFrame(rows)
    results = results.sort_values(["Seed", "Model"]).reset_index(drop=True)

    result_path = WORK / "weedy_seed123_2026_results.csv"
    results.to_csv(result_path, index=False)

    print("\n" + "=" * 100)
    print("FINAL RESULTS — SEEDS 123 AND 2026 ONLY")
    print("=" * 100)
    print(
        results.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}",
        )
    )
    print("\nSaved:", result_path)


print("\n" + "=" * 100)
print("ALL SIX TRAINING RUNS FINISHED")
print("=" * 100)
